# Faza 2 — Test kloniranja glasa (OpenAudio S1-mini / Fish Speech)

Klonira tvoj glas iz `owner-sample.wav` i izgovori **srpski** probni pasus. Ti preslušaš rezultat i odlučujemo da li je dovoljno dobro (OQ1 iz PRD-a).

**Pre početka:** meni *Runtime → Change runtime type → Hardware accelerator: **GPU*** (besplatan T4 je dovoljan). Sve ovde je besplatno na free Colab GPU.

Tok ima 3 koraka (zvanična dokumentacija: https://speech.fish.audio/inference/):
1. enkoduj referentni glas → `fake.npy`
2. generiši semantičke tokene iz teksta (uslovljeno referencom) → `codes_0.npy`
3. dekoduj tokene u zvuk → `fake.wav`

## 1. Proveri GPU

In [ ]:
!nvidia-smi

## 2. Poveži Google Drive (ulaz/izlaz)

Prebaci `owner-sample.wav` na svoj Drive, npr. u folder `ai-glas/`. Odavde ga čitamo i tu vraćamo rezultat. (Alternativa bez Drive-a je dole u komentaru — direktan upload.)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/ai-glas'   # PRILAGODI ako koristiš drugi folder
os.makedirs(DRIVE_DIR, exist_ok=True)
REF_FULL = f'{DRIVE_DIR}/owner-sample.wav'
print('Reference:', REF_FULL, '->', 'OK' if os.path.exists(REF_FULL) else 'NIJE NAĐEN — prebaci fajl na Drive')

# --- Alternativa bez Drive-a (otkomentariši): ---
# from google.colab import files; up = files.upload()  # izaberi owner-sample.wav
# REF_FULL = '/content/' + list(up.keys())[0]

## 3. Instaliraj Fish Speech

Kloniramo repo i instaliramo paket. (Ako pip prijavi sukob torch verzija, obično svejedno radi — nastavi dalje; ako pukne inferencija, vidi *Rešavanje problema* na dnu.)

*Napomena:* posle instalacije pip ispise gomilu upozorenja o verzijama (protobuf, torchvision, google-cloud...). To je ocekivano i bezopasno — ti paketi se ovde ne koriste. Bitno je samo da torch vidi GPU, sto proveravamo odmah ispod.

In [ ]:
%cd /content
!git clone https://github.com/fishaudio/fish-speech.git 2>/dev/null || echo '(repo vec postoji)'
%cd /content/fish-speech
# portaudio je potreban da se izgradi pyaudio (inace pip pukne); pyaudio je za mikrofon, ne za inferenciju
!apt-get -qq install -y portaudio19-dev
!pip install -e . -q
# Fish Speech spusti torch na 2.8.0; uskladi torchvision (inace puca: torchvision::nms does not exist)
!pip install -q torchvision==0.23.0
# openaudio-s1-mini tokenizer trazi transformers<=4.57.3 (inace: NoneType ... encode)
!pip install -q "transformers==4.57.3"
print('Instalirano. Skripte za inferenciju:')
!ls fish_speech/models/dac/inference.py fish_speech/models/text2semantic/inference.py

### Provera okruženja (pre nastavka)
Gornja pip upozorenja su normalna. Pokreni proveru — mora da pise **CUDA dostupna: True**.

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA dostupna:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU nije aktivan: Runtime -> Change runtime type -> GPU, pa Runtime -> Restart i pokreni ispocetka.'

## 4. Preuzmi model (OpenAudio S1-mini, 0.5B — otvoren i besplatan)

Ako HuggingFace traži saglasnost: otvori https://huggingface.co/fishaudio/openaudio-s1-mini , prihvati uslove, pa otkomentariši `login()` ispod i nalepi svoj **besplatni** HF token (Settings → Access Tokens).

In [ ]:
# from huggingface_hub import login; login()   # otkomentariši ako traži token

MODEL_DIR = 'checkpoints/openaudio-s1-mini'
CODEC = f'{MODEL_DIR}/codec.pth'
!huggingface-cli download fishaudio/openaudio-s1-mini --local-dir {MODEL_DIR}
print('Model:', MODEL_DIR)
!ls -la {MODEL_DIR}

## 5. Napravi kratak referentni isečak (~22s) + njegov tačan transkript

Fish Speech klonira iz **kratkog** uzorka (10–30s) + **teksta koji se u njemu govori**. Pravimo prvih 22 sekunde tvog snimka, ti ga preslušaš i upišeš tačno šta se čuje.

In [ ]:
REF = '/content/ref22.wav'
!ffmpeg -y -i "{REF_FULL}" -t 22 -ac 1 -ar 44100 "{REF}" -loglevel error
import IPython.display as ipd
print('Preslušaj isečak, pa u sledećoj ćeliji upiši TAČAN tekst:')
ipd.Audio(REF)

In [ ]:
# >>> Upiši TAČNO ono što se čuje u ref22.wav (prvih ~22s tvog snimka).
# Pred-popunjeno po početku RECORDING_SCRIPT.md — ISPRAVI ako si čitao drugim redom!
PROMPT_TEXT = "Veštačka inteligencija danas više nije nešto što gledamo samo u filmovima. Ona piše tekst, pravi slike, pomaže u programiranju i odgovara na pitanja iz skoro svake oblasti."

# >>> Šta želimo da model izgovori TVOJIM glasom (rečenica iz scenarija videa 001):
TARGET_TEXT = "Veštačka inteligencija u 2026. više nije naučna fantastika. Danas ti piše kod, pravi slike i odgovara na pitanja bolje nego ikad."
print('PROMPT_TEXT i TARGET_TEXT postavljeni.')

## 6. Korak 1/3 — enkoduj referentni glas u tokene

Pravi `fake.npy` (otisak tvog glasa).

In [ ]:
!python fish_speech/models/dac/inference.py -i "{REF}" --checkpoint-path "{CODEC}"
!ls -la fake.npy

## 7. Korak 2/3 — generiši semantičke tokene iz srpskog teksta

Uslovljeno tvojim glasom (`fake.npy`) i transkriptom reference. Pravi `codes_0.npy`. (Dodaj `--half` ako GPU nema bf16 / javi OOM.)

In [ ]:
!python fish_speech/models/text2semantic/inference.py --text "{TARGET_TEXT}" --prompt-text "{PROMPT_TEXT}" --prompt-tokens "fake.npy" --checkpoint-path "{MODEL_DIR}" --num-samples 1
!ls -la codes_0.npy

## 8. Korak 3/3 — dekoduj tokene u zvuk

Pravi finalni `fake.wav` (tvoj klonirani glas govori srpski).

In [ ]:
!python fish_speech/models/dac/inference.py -i "codes_0.npy" --checkpoint-path "{CODEC}"
!ls -la fake.wav

## 9. Preslušaj i sačuvaj na Drive

In [ ]:
import shutil, IPython.display as ipd
OUT = '/content/fish-speech/fake.wav'
try:
    shutil.copy(OUT, f'{DRIVE_DIR}/clone-test-fish.wav')
    print('Sačuvano na Drive:', f'{DRIVE_DIR}/clone-test-fish.wav')
except Exception as e:
    print('Nisam sačuvao na Drive (', e, ') — svejedno možeš preslušati ispod.')
ipd.Audio(OUT)

## Šta slušamo (kriterijum)
- Da li zvuči kao **tvoj glas**?
- Da li je **srpski izgovor** prirodan (č/ć/đ/š/ž, akcenat, ‘2026.’ kao ‘dve hiljade dvadeset šesta’)?
- Robotski / artefakti / pogrešan akcenat?

Ako je dobro → idemo na punu naraciju + poravnanje (`02-voice`).
Ako je loše na srpskom → probamo XTTS-v2 (zaseban notebook) i/ili presnimiš duži, glasniji uzorak.

---
## Rešavanje problema
- **`huggingface-cli: command not found`** → `!pip install -U huggingface_hub` pa probaj `!hf download fishaudio/openaudio-s1-mini --local-dir checkpoints/openaudio-s1-mini`.
- **Gated/403 pri preuzimanju** → prihvati uslove na stranici modela + `login()` sa HF tokenom.
- **Putanja skripte ne postoji** → `!ls fish_speech/models` i `!ls tools` da vidiš tačan raspored za instaliranu verziju; uskladi komande.
- **OOM / CUDA** → dodaj `--half` u korak 2; smanji dužinu TARGET_TEXT; Runtime → Restart.
- **Sukob torch verzija** → prati šta `pip` predlaže; često radi i uz upozorenje.
- **Interaktivno (opciono):** umesto koraka 6–8 možeš `!python tools/run_webui.py` i koristiti Gradio UI (vidi `scripts/colab/README.md`).